<a href="https://colab.research.google.com/github/mateosanlu/DeepLearning/blob/main/DL_Laboratorio_Transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laboratorio: Transformers

## Leidy Mireya Torres Romero - Mateo Sandoval Luna

Este informe documenta la implementación de un traductor neuronal bidireccional español–inglés basado en modelos Transformer preentrenados de la familia OPUS-MT (MarianMT), junto con un componente de reconocimiento óptico de caracteres (OCR) para traducir texto contenido en imágenes.

## Paso 1. Instalación de dependencias

Esta fase instala las bibliotecas que requiere la aplicación. Transformers permite cargar modelos de traducción preentrenados; PyTorch ejecuta los modelos; EasyOCR reconoce texto en imágenes; Gradio construye la interfaz web. SentencePiece y Sacremoses son dependencias utilizadas por los modelos OPUS-MT.

In [2]:
!pip install -q gradio transformers sentencepiece sacremoses easyocr

## Paso 2. Importación de librerías y selección del dispositivo

Se importan las bibliotecas que intervienen en la práctica. Luego PyTorch comprueba si el entorno dispone de CUDA. Si hay GPU compatible, se selecciona para acelerar la inferencia; de lo contrario, el procesamiento se realiza en CPU. La variable `device` se reutiliza al cargar los modelos y al preparar los tensores.

In [3]:
import torch
import gradio as gr
import easyocr
import numpy as np
from PIL import Image
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Ejecutando en: {device.upper()}")

🚀 Ejecutando en: CPU


## Paso 3. Carga del modelo Español a Inglés

Se carga el modelo `Helsinki-NLP/opus-mt-es-en`. `AutoTokenizer` prepara el texto de entrada convirtiéndolo en tokens y `AutoModelForSeq2SeqLM` carga el modelo neuronal que genera la secuencia traducida. `.to(device)` ubica el modelo en el dispositivo seleccionado.

In [4]:
print("⏳ Cargando modelo Español ➔ Inglés...")
model_name_es_en = "Helsinki-NLP/opus-mt-es-en"
tokenizer_es_en = AutoTokenizer.from_pretrained(model_name_es_en)
model_es_en = AutoModelForSeq2SeqLM.from_pretrained(model_name_es_en).to(device)

⏳ Cargando modelo Español ➔ Inglés...


config.json:   0%|          | 0.00/1.44k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/826k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.59M [00:00<?, ?B/s]

... (2 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B /  312MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

## Paso 4. Carga del modelo Inglés a Español

Se carga el modelo `Helsinki-NLP/opus-mt-en-es`, que realiza la traducción inversa. Se conservan objetos separados de tokenizador y modelo para cada dirección, ya que cada modelo está especializado en su par de idiomas.

In [5]:
print("⏳ Cargando modelo Inglés ➔ Español...")
model_name_en_es = "Helsinki-NLP/opus-mt-en-es"
tokenizer_en_es = AutoTokenizer.from_pretrained(model_name_en_es)
model_en_es = AutoModelForSeq2SeqLM.from_pretrained(model_name_en_es).to(device)

⏳ Cargando modelo Inglés ➔ Español...


config.json:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/826k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.59M [00:00<?, ?B/s]

... (2 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B /  312MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

## Paso 5. Inicialización del motor OCR

EasyOCR se inicializa para reconocer texto en inglés y español. La opción `gpu` activa el procesamiento mediante GPU cuando CUDA está disponible. El lector se utiliza después para detectar y extraer texto de las imágenes cargadas por el usuario.

In [6]:
print("⏳ Inicializando detector OCR...")
ocr_reader = easyocr.Reader(['en', 'es'], gpu=(device == "cuda"))
print("✅ ¡Modelos listos!")

⏳ Inicializando detector OCR...
Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

/usr/local/lib/python3.13/dist-packages/torch/ao/nn/quantized/dynamic/modules/rnn.py:162: UserWarning: torch.quantize_per_tensor, torch.quantize_per_channel and other quantized tensor creation functions that produce tensors with dtype torch.quint8, torch.qint8, and torch.qint32 are deprecated and will be removed in a future PyTorch release. Please see https://github.com/pytorch/pytorch/issues/184982 for more information. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/quantized/Quantizer.cpp:111.)
  w_ih = torch.quantize_per_tensor(


✅ ¡Modelos listos!


## Paso 6. Función para traducir texto

`traducir_texto` recibe el contenido y la dirección de traducción. Valida que la entrada no esté vacía, selecciona el modelo y tokenizador adecuados, convierte el texto en tensores, genera la secuencia de salida y la decodifica como texto legible. `torch.no_grad()` indica que se realiza inferencia y no entrenamiento.

In [7]:
def traducir_texto(texto: str, direccion: str) -> str:
    if not texto or not texto.strip():
        return "⚠️ Por favor, ingresa un texto para traducir."


    if direccion == "Español ➔ Inglés":
        tok, mod = tokenizer_es_en, model_es_en
    else:
        tok, mod = tokenizer_en_es, model_en_es


    inputs = tok(texto, return_tensors="pt", padding=True, truncation=True).to(device)


    with torch.no_grad():
        salida_ids = mod.generate(**inputs, max_length=512)


    return tok.decode(salida_ids[0], skip_special_tokens=True)

## Paso 7. Función de OCR y traducción de imágenes

`traducir_imagen` recibe una imagen PIL y la dirección de traducción. La convierte en una matriz NumPy, ejecuta EasyOCR y une los fragmentos reconocidos. Si no encuentra texto, devuelve un mensaje; si lo encuentra, llama a `traducir_texto` y devuelve el texto extraído junto con su traducción.

In [8]:
def traducir_imagen(imagen: Image.Image, direccion: str):
    if imagen is None:
        return "⚠️ No se ha subido ninguna imagen.", ""


    img_np = np.array(imagen)
    textos_detectados = ocr_reader.readtext(img_np, detail=0)
    texto_extraido = " ".join(textos_detectados).strip()

    if not texto_extraido:
        return "🔍 No se detectó texto en la imagen.", ""


    traduccion = traducir_texto(texto_extraido, direccion)
    return texto_extraido, traduccion

## Paso 8. Construcción de la interfaz con Gradio

Se construye la aplicación web mediante `gr.Blocks`. La primera pestaña permite ingresar texto, elegir el sentido de traducción y consultar el resultado. La segunda permite cargar una imagen y muestra el texto detectado por OCR y la traducción. Los eventos `.click()` conectan los botones con las funciones creadas en los pasos 6 y 7.

In [9]:
with gr.Blocks(theme=gr.themes.Soft(), title="Traductor IA") as demo:
    gr.Markdown("# 🌐 Traductor Multimodal de Texto e Imágenes (EN ↔ ES)")

    with gr.Tabs():

        with gr.TabItem("📝 Traducción de Texto"):
            txt_in = gr.Textbox(label="Ingresa texto", placeholder="Escribe aquí...", lines=4)
            dir_txt = gr.Radio(["Español ➔ Inglés", "Inglés ➔ Español"], value="Español ➔ Inglés", label="Dirección")
            btn_txt = gr.Button("Traducir Texto", variant="primary")
            txt_out = gr.Textbox(label="Traducción", lines=4, interactive=False)

            btn_txt.click(traducir_texto, inputs=[txt_in, dir_txt], outputs=txt_out)


        with gr.TabItem("📷 Traductor desde Imagen (OCR)"):
            img_in = gr.Image(type="pil", label="Sube una foto con texto en inglés")
            dir_img = gr.Radio(["Inglés ➔ Español", "Español ➔ Inglés"], value="Inglés ➔ Español", label="Dirección de traducción")
            btn_img = gr.Button("Escanear y Traducir", variant="primary")
            ocr_out = gr.Textbox(label="1. Texto detectado por OCR", lines=2, interactive=False)
            img_tr_out = gr.Textbox(label="2. Traducción al Español", lines=3, interactive=False)

            btn_img.click(traducir_imagen, inputs=[img_in, dir_img], outputs=[ocr_out, img_tr_out])

/tmp/ipykernel_634/2227155980.py:1: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Traductor IA") as demo:


## Paso 9. Lanzamiento de la aplicación y generación del enlace

Esta celda inicia la interfaz y solicita a Gradio crear un enlace público temporal. Después de ejecutar el código, espera a que termine el arranque y busca en la salida de la celda la línea **Running on public URL**. Abre ese enlace para acceder al traductor desde el navegador. Si no aparece, verifica que la celda haya terminado correctamente y que el entorno de Colab tenga conexión a Internet.

In [10]:
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4135a276006f63936b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Objetivo alcanzado y comprensión de Transformers

La práctica permite reconocer el flujo básico de inferencia de un modelo Transformer de traducción: seleccionar un modelo preentrenado, preparar el texto mediante un tokenizador, generar una secuencia con el modelo y decodificarla para obtener el resultado. También muestra cómo integrar este proceso con un componente de visión por computador (OCR) y exponerlo mediante una interfaz web.

## Conclusiones

1. **Transformers facilita el uso de modelos de traducción preentrenados** sin tener que construir ni entrenar desde cero una arquitectura neuronal. La práctica permite identificar la función diferenciada del tokenizador y del modelo de secuencia a secuencia: el primero convierte el texto en entradas numéricas y el segundo genera la secuencia traducida.

2. **La integración de OCR amplía la aplicación del modelo más allá del texto escrito manualmente.** EasyOCR extrae el contenido de una imagen y lo entrega al mismo proceso de traducción. De esta forma, se observa cómo distintos componentes de inteligencia artificial pueden combinarse en una solución funcional.

## Recomendaciones

- Probar frases cortas, textos extensos y expresiones con vocabulario técnico en ambas direcciones para revisar la calidad de la traducción. La calidad puede variar según el contexto y el dominio del texto.
- Evaluar el OCR con imágenes de distinta resolución, iluminación, orientación y tipografía. Revisar el texto detectado antes de traducirlo, especialmente cuando se trate de documentos importantes.
- Registrar ejemplos de entrada y salida para comparar resultados y detectar errores de traducción o reconocimiento. El notebook original no incluye resultados de ejecución guardados, por lo que se recomienda ejecutar las pruebas y documentar ejemplos verificables.
- Considerar que `share=True` crea un enlace público temporal. Evitar cargar imágenes con información personal, confidencial o institucional mientras se utilice esa opción.
- Para una versión posterior, incorporar manejo de excepciones durante la descarga de modelos y el procesamiento de imágenes, así como mostrar al usuario el idioma detectado y las limitaciones de la traducción.